
#  Layer 1: Raw / Landing Layer (CSV-Based)

**Purpose:**

* Store **raw, unprocessed CSV data** exactly as received from source systems.
* No transformations, joins, or calculations.
* Ensure **immutability, traceability, and re-load safety**.

---

## 1️ Key Characteristics

| Characteristic               | Description                                                          |
| ---------------------------- | -------------------------------------------------------------------- |
| Data Type                    | STRING (all fields)                                                  |
| Source                       | CSV files (customers.csv, transactions.csv, products.csv)            |
| Transformations              | None – only metadata added                                           |
| Metadata                     | Source filename, load timestamp                                      |
| Layer Role                   | Acts as the **single source of truth** for all downstream processing |
| Reprocessing Support         | ✅ Full – can reload CSVs anytime                                     |
| Failures & Bad Data Handling | Minimal risk because all fields are strings                          |

---

## 2️ Snowflake Objects

### 2.1 Warehouse

```sql
CREATE OR REPLACE WAREHOUSE ingest_wh
WITH
    WAREHOUSE_SIZE = 'XSMALL'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE;
USE WAREHOUSE ingest_wh;
```

* Dedicated compute for ingestion
* Isolated from analytics workloads
* Cost-efficient (auto-suspend/resume)

---

### 2.2 Schemas

```sql
CREATE OR REPLACE DATABASE sales_analytics;
CREATE OR REPLACE SCHEMA sales_analytics.raw;
CREATE OR REPLACE SCHEMA sales_analytics.staging;
CREATE OR REPLACE SCHEMA sales_analytics.production;

USE DATABASE sales_analytics;
```

---

### 2.3 File Format (CSV)

```sql
CREATE OR REPLACE FILE FORMAT csv_ff
TYPE = 'CSV'
FIELD_DELIMITER = ','
SKIP_HEADER = 1
TRIM_SPACE = TRUE
FIELD_OPTIONALLY_ENCLOSED_BY = '"'
NULL_IF = ('NULL', 'null', '');
```

* Handles headers, quotes, whitespace, and missing values

---

### 2.4 Stage (Landing Area)

```sql
CREATE OR REPLACE STAGE sales_csv_stage
FILE_FORMAT = csv_ff;
```

* Internal Snowflake stage for CSV uploads
* Upload CSVs like:

```
PUT file://customers.csv     @sales_csv_stage;
PUT file://transactions.csv  @sales_csv_stage;
PUT file://products.csv      @sales_csv_stage;
```

---

### 2.5 Raw Tables

| Table            | Columns (all STRING except load_ts)                                                                   |
| ---------------- | ----------------------------------------------------------------------------------------------------- |
| raw.customers    | customer_id, customer_name, email, country, created_date, source_file, load_ts                        |
| raw.transactions | transaction_id, customer_id, product_id, quantity, unit_price, transaction_date, source_file, load_ts |
| raw.products     | product_id, product_name, category, supplier, unit_price, source_file, load_ts                        |

**Example Table Creation (Customers):**

```sql
CREATE OR REPLACE TABLE raw.customers (
    customer_id   STRING,
    customer_name STRING,
    email         STRING,
    country       STRING,
    created_date  STRING,
    source_file   STRING,
    load_ts       TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP
);
```

---

### 2.6 Load Data (COPY INTO)

**Example: Load Customers CSV**

```sql
COPY INTO raw.customers
FROM (
    SELECT $1, $2, $3, $4, $5, METADATA$FILENAME
    FROM @sales_csv_stage/customers.csv
)
FILE_FORMAT = csv_ff;
```

* Similar commands for `transactions` and `products`

---

### 2.7 Validation Queries

```sql
-- Check row counts
SELECT COUNT(*) FROM raw.customers;
SELECT COUNT(*) FROM raw.transactions;
SELECT COUNT(*) FROM raw.products;

-- Preview data
SELECT * FROM raw.transactions LIMIT 5;
```

* Confirms CSVs landed correctly
* Ensures strings are preserved for safe downstream transformation

---

## 3️ Best Practices / Notes

* **All fields as STRING:** avoids load failures due to bad data
* **Metadata captured:** filename + load timestamp for traceability
* **No business logic here:** preserves raw source for auditing/reprocessing
* **Re-load friendly:** CSVs can be re-ingested without impacting analytics

---

###  Interview Explanation

> “Layer 1 is our landing layer where we ingest CSV files into Snowflake raw tables. All fields are stored as strings with metadata like source filename and load timestamp. No transformations are applied here, which ensures immutability, auditability, and safe reprocessing. This layer acts as the single source of truth for downstream staging and production layers.”

---

✅ **Layer 1 Complete:**

* Warehouse ready for ingestion
* Raw CSV tables created
* Internal stage for CSVs
* COPY INTO commands for loading
* Validation queries for row counts and preview
